# CKP02 — DocMind RAG · Suporte Técnico de Aparelhos Eletrônicos
**Prompt Engineering and Artificial Intelligence · FIAP — Ciência da Computação · 2026**

Grupo 02 — integrantes e RMs estão no `README.md` do projeto.

Pipeline completo: **load → split → embed → store → retrieve → generate**, com comparação de chunking via RAGAS,
metadata filtering, reranking e interface Gradio. Este notebook usa os mesmos módulos da pasta `app/` do projeto
(`rag_service.py`, `base_conhecimento.py`, `avaliar_rag.py`), então o que é medido aqui é o que o chatbot entrega.

**Antes de rodar:** (1) salve `OLLAMA_API_KEY` em *Secrets* do Colab; (2) tenha em mãos o `.zip` do projeto.

### 1. Instalação e configuração (Ollama Cloud)

In [ ]:
!pip install -q langchain-ollama langchain-core langchain-classic langchain-chroma chromadb langchain-text-splitters langchain-community pymupdf sentence-transformers gradio ragas datasets python-dotenv pandas numpy tiktoken

import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

### 2. Projeto e documentos da base
Envie o `.zip` do projeto (ele contém `app/` e `docs/`).

In [ ]:
import glob, os, sys, zipfile
from google.colab import files

if not os.path.exists("app/rag_service.py"):
    enviado = files.upload()  # selecione o .zip do projeto
    for nome in enviado:
        with zipfile.ZipFile(nome) as z:
            z.extractall("projeto")
    raiz = os.path.dirname(os.path.dirname(glob.glob("projeto/**/app/rag_service.py", recursive=True)[0]))
    os.chdir(raiz)

sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())
print("Documentos:", sorted(os.listdir("docs")))

### 3. Fontes dos documentos (base real, com origem citada)

In [ ]:
import json
import pandas as pd

catalogo = pd.DataFrame(json.load(open("docs/catalogo.json", encoding="utf-8")))
pd.set_option("display.max_colwidth", 120)
catalogo[["arquivo", "produto", "categoria", "tipo", "fonte_url"]]

### 4. LOAD — carregar documentos com metadados

In [ ]:
from app.base_conhecimento import carregar_documentos, listar_categorias

documentos = carregar_documentos("docs")
print(f"Documentos carregados: {len(documentos)}")
print(f"Categorias: {listar_categorias(documentos)}")
print("Metadados do 1º documento:", documentos[0].metadata)
print("\nTrecho:\n", documentos[0].page_content[:300])

### 5. SPLIT — três configurações de chunk_size (overlap = 12,5%)

In [ ]:
import statistics
from langchain_text_splitters import RecursiveCharacterTextSplitter

for tamanho in [256, 512, 1024]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=tamanho,
        chunk_overlap=tamanho // 8,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documentos)
    tamanhos = [len(c.page_content) for c in chunks]
    print(f"chunk_size={tamanho:4d} | chunks={len(chunks):3d} | média={statistics.mean(tamanhos):.0f} chars "
          f"| mín/máx={min(tamanhos)}/{max(tamanhos)}")

### 6. EMBED — nomic-embed-text via Ollama Cloud

In [ ]:
import numpy as np
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vetor = embeddings.embed_query("Como reiniciar o roteador?")
print(f"Dimensões: {len(vetor)} · tipo: {type(vetor[0]).__name__}")

def similaridade_cosseno(v1, v2) -> float:
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

v_perto = embeddings.embed_query("Restaurar o roteador para as configurações de fábrica")
v_longe = embeddings.embed_query("Receita de bolo de cenoura")
print(f"consulta × frase do domínio : {similaridade_cosseno(vetor, v_perto):.4f}")
print(f"consulta × frase fora       : {similaridade_cosseno(vetor, v_longe):.4f}")

### 7. STORE + RETRIEVE — ChromaDB (coleção com o nome do domínio) e busca semântica

In [ ]:
from app.rag_service import SuporteRAGService, formatar_fontes

servicos = {tam: SuporteRAGService(docs_dir="docs", chunk_size=tam) for tam in [256, 512, 1024]}
rag = servicos[1024]
print("Coleção:", rag.nome_colecao, "| chunks indexados:", rag.vectorstore._collection.count())

for doc, similaridade, _ in rag.recuperar("a TV está sem imagem, o que fazer?"):
    print(f"[{similaridade:.3f}] {doc.metadata['arquivo']} (trecho {doc.metadata['trecho']}): {doc.page_content[:90]}...")

### 8. Metadata filtering — `where=` antes da busca

In [ ]:
from app.rag_service import montar_filtro

consulta = "como restaurar as configurações de fábrica"
print("SEM filtro:")
for doc, sim, _ in rag.recuperar(consulta):
    print(f"  [{sim:.3f}] {doc.metadata['categoria']:10s} {doc.metadata['arquivo']}")

print("\nCOM filtro categoria=roteador:")
for doc, sim, _ in rag.recuperar(consulta, filtro=montar_filtro(categoria="roteador")):
    print(f"  [{sim:.3f}] {doc.metadata['categoria']:10s} {doc.metadata['arquivo']}")

print("\nCOM filtro composto ($and) categoria=roteador + marca=TP-Link:")
for doc, sim, _ in rag.recuperar(consulta, filtro=montar_filtro(categoria="roteador", marca="TP-Link")):
    print(f"  [{sim:.3f}] {doc.metadata['marca']:10s} {doc.metadata['arquivo']}")

### 9. Reranking — cross-encoder `ms-marco-MiniLM-L-6-v2` (busca ampla k=10 → top 3)

In [ ]:
consulta = "Como recuperar a senha do Wi-Fi?"
print("SEM reranking (ordem da distância cosseno):")
for doc, sim, _ in rag.recuperar(consulta, usar_reranking=False):
    print(f"  [{sim:.3f}] {doc.metadata['arquivo']} (trecho {doc.metadata['trecho']})")

print("\nCOM reranking (cross-encoder reordena):")
for doc, sim, nota in rag.recuperar(consulta, usar_reranking=True):
    print(f"  [rerank {nota:.3f} | cosseno {sim:.3f}] {doc.metadata['arquivo']} (trecho {doc.metadata['trecho']})")

### 10. GENERATE — pipeline end-to-end com fonte citada

In [ ]:
from app.avaliar_rag import PERGUNTAS

for pergunta in PERGUNTAS:
    resultado = rag.consultar(pergunta)
    print(f"\n📌 {pergunta}\n{resultado['resposta']}")
    print(formatar_fontes(resultado["fontes"], tamanho_trecho=90))

### 11. RAGAS — faithfulness e answer_relevancy por pergunta e por configuração
Usa o mesmo LLM (`gemma4:cloud`, `temperature=0`) como juiz. Compara 256, 512 e 1024 (as mesmas configurações da demo da Aula 07). Pode levar alguns minutos.

In [ ]:
from app.avaliar_rag import comparar, salvar_resultados, FAITHFULNESS_APROVACAO

tabelas, resumos, vencedora = comparar([256, 512, 1024])
salvar_resultados(tabelas, resumos, vencedora)

pd.concat(tabelas).round(3)

### 12. Comparação final e escolha da configuração

In [ ]:
resumo_df = pd.DataFrame(resumos).round(3)
display(resumo_df)

print(f"Configuração vencedora: chunk_size = {vencedora['chunk_size']}")
print(f"Faithfulness = {vencedora['faithfulness']:.3f} "
      f"({'≥' if vencedora['faithfulness'] >= FAITHFULNESS_APROVACAO else '<'} meta de {FAITHFULNESS_APROVACAO})")
print(f"Answer relevancy = {vencedora['answer_relevancy']:.3f}")
print("\nObs.: a pergunta fora do domínio (geladeira) é recusada de propósito e recebe answer_relevancy 0 no RAGAS, "
      "o que reduz a média dessa métrica.")

### 12.1 Análise de falhas — por que a pior pergunta errou?
Pega a pergunta com menor faithfulness da configuração vencedora e mostra os chunks que o retriever entregou ao LLM (Aula 07, Exercício 4).

Regra de leitura: **trecho presente** no contexto e resposta ruim → problema no prompt/LLM; **trecho ausente** → problema no retriever (chunk_size, overlap ou k).

In [ ]:
from app.avaliar_rag import diagnosticar_pior_pergunta

diag = diagnosticar_pior_pergunta(tabelas, servicos[vencedora["chunk_size"]], vencedora["chunk_size"])

### 12.2 Resultado e justificativa (gerados a partir dos números reais)

In [ ]:
from IPython.display import Markdown, display
from app.atualizar_readme import gerar_bloco, _ler_csv
from app.config_rag import carregar_vencedora

display(Markdown(gerar_bloco(carregar_vencedora(), _ler_csv())))

### 12.3 Análise do grupo — qual configuração ganhou e por quê
**Preencha esta célula depois de rodar a seção 11** (o critério do professor exige justificar a escolha com os dados):

- **Vencedora:** `chunk_size = ___` (faithfulness ___ · answer_relevancy ___).
- **Por que ganhou (hipótese do grupo):** ___ (ex.: chunks pequenos demais cortam instruções passo a passo dos manuais; chunks grandes diluem o embedding com assuntos vizinhos).
- **Onde errou:** a pergunta ___ teve a menor nota porque ___ (veja a seção 12.1).
- **Limitação:** a pergunta fora do domínio (geladeira) é recusada de propósito e recebe answer_relevancy 0, o que reduz a média dessa métrica.

### 13. Função modular `buscar(consulta)` — pronta para virar `@tool` no CKP03
Usa a configuração vencedora do RAGAS e o índice persistido em `chroma_db/`.

In [ ]:
from app.rag_service import buscar

print(buscar("Qual é o usuário e a senha padrão do roteador Intelbras WRN 240?"))
print()
print(buscar("Como restaurar as configurações de fábrica?", categoria="roteador", usar_reranking=True))

### 14. Interface Gradio (diferencial)

In [ ]:
from app.interface_gradio import criar_interface

demo = criar_interface(rag)
demo.launch(share=True)  # gera link público temporário

## Conclusão
A escolha do `chunk_size` está justificada pelos números das seções 11 e 12 (tabela por pergunta, análise de falhas e arquivos em `resultados/`).
Para adicionar novos documentos à base, veja a seção *Como adicionar novos documentos* do `README.md`.

### Checklist final de entrega (rubrica do CKP02)
Confere automaticamente base, pipeline, RAGAS e README. Itens de notebook executado só ficam OK depois de exportar o `.ipynb` com as saídas.

In [ ]:
from app.verificar_entrega import verificar

for ok, requisito, detalhe in verificar():
    print(f"[{'OK' if ok else '  '}] {requisito}" + (f"  → {detalhe}" if detalhe else ""))